# Exploraci?n de trayectorias de taxis de Porto

Este notebook explora una muestra reproducible de `train.csv`: valida las trayectorias y calcula variables temporales, distancia y velocidad. El procesamiento completo del dataset debe hacerse con Dask en el pipeline del proyecto, no cargando casi 2 GB en memoria con pandas.

Las coordenadas de `POLYLINE` est?n en orden `[longitud, latitud]`. Las variables de calendario se derivan en UTC. Las distancias se calculan como aproximaciones sobre la esfera terrestre.

# Imports y funciones de validaci?n

In [6]:
import json
from pathlib import Path

import numpy as np
import pandas as pd


def parse_polyline(value):
    if not isinstance(value, str) or not value.strip():
        return None

    try:
        points = json.loads(value)
    except json.JSONDecodeError:
        return None

    if not isinstance(points, list) or not points:
        return None

    parsed_points = []
    for point in points:
        if not isinstance(point, list) or len(point) != 2:
            return None

        lon, lat = point
        if (
            isinstance(lon, bool)
            or isinstance(lat, bool)
            or not isinstance(lon, (int, float))
            or not isinstance(lat, (int, float))
            or not np.isfinite(lon)
            or not np.isfinite(lat)
            or not -180 <= lon <= 180
            or not -90 <= lat <= 90
        ):
            return None

        parsed_points.append((float(lon), float(lat)))

    return parsed_points

# Carga de una muestra local

In [7]:
# El CSV ya descargado puede estar en data/ (ejecuci?n desde la ra?z del proyecto)
# o en la carpeta actual (ejecuci?n desde data/).
data_candidates = [Path("data/train.csv"), Path("train.csv"), Path("../data/train.csv")]
DATA_PATH = next((candidate for candidate in data_candidates if candidate.is_file()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "No se encontr? train.csv. Col?calo en data/train.csv o ejecuta el notebook desde data/."
    )

SAMPLE_ROWS = 10_000
taxi = pd.read_csv(DATA_PATH, nrows=SAMPLE_ROWS, low_memory=False)
print(f"Archivo: {DATA_PATH.resolve()}")
print(f"Filas cargadas para exploraci?n: {len(taxi):,}")

Archivo: D:\Uni\BigData\Parcial\data\train.csv
Filas cargadas para exploraci?n: 10,000


# Variables temporales (UTC)

In [8]:
taxi["TIMESTAMP"] = pd.to_numeric(taxi["TIMESTAMP"], errors="coerce")
taxi = taxi.loc[taxi["TIMESTAMP"].notna()].copy()
taxi["TIMESTAMP"] = taxi["TIMESTAMP"].astype("int64")

trip_start_utc = pd.to_datetime(taxi["TIMESTAMP"], unit="s", utc=True)
taxi["YEAR"] = trip_start_utc.dt.year
taxi["MONTH"] = trip_start_utc.dt.month
taxi["DAY"] = trip_start_utc.dt.day
taxi["HOUR"] = trip_start_utc.dt.hour
taxi["WEEKDAY"] = trip_start_utc.dt.weekday

taxi[["TIMESTAMP", "YEAR", "MONTH", "DAY", "HOUR", "WEEKDAY"]].head()

,TIMESTAMP,YEAR,MONTH,DAY,HOUR,WEEKDAY
0,1372636858,2013,7,1,0,0
1,1372637303,2013,7,1,0,0
2,1372636951,2013,7,1,0,0
3,1372636854,2013,7,1,0,0
4,1372637091,2013,7,1,0,0


# Limpieza

In [9]:
len(taxi)

10000

## Eliminaci?n de duplicados

In [10]:
taxi = taxi.drop_duplicates().copy()

In [11]:
len(taxi)

10000

## Filtrado de viajes con datos faltantes

In [12]:
taxi = taxi.loc[
    taxi["MISSING_DATA"].astype("string").str.strip().str.lower().eq("false")
].copy()

In [13]:
len(taxi)

10000

## Validaci?n de polil?neas y coordenadas

In [14]:
taxi["POINTS"] = taxi["POLYLINE"].map(parse_polyline)
valid_polyline = taxi["POINTS"].map(lambda points: points is not None)
invalid_count = int((~valid_polyline).sum())
taxi = taxi.loc[valid_polyline].copy()
print(f"Trayectorias vac?as, inv?lidas o fuera de rango descartadas: {invalid_count:,}")

Trayectorias vac?as, inv?lidas o fuera de rango descartadas: 40


In [15]:
len(taxi)

9960

## Reinicio del ?ndice

In [16]:
taxi.reset_index(drop=True, inplace=True)

# Origen y destino del viaje

Cada par est? en orden `[longitud, latitud]`. El punto de origen se usar? como la geometr?a principal en el modelo del proyecto; el destino se conserva como atributo separado.

In [17]:
def map_trip_endpoints(df_in):
    points = df_in["POINTS"]
    return pd.DataFrame(
        {
            "TRIP_START": points.map(lambda trajectory: trajectory[0]),
            "TRIP_END": points.map(lambda trajectory: trajectory[-1]),
        },
        index=df_in.index,
    )

In [18]:
endpoint_columns = map_trip_endpoints(taxi)
taxi = pd.concat([taxi, endpoint_columns], axis=1)
taxi[["TRIP_START", "TRIP_END"]].head()

,TRIP_START,TRIP_END
0,"(-8.618643, 41.141412)","(-8.630838, 41.154489)"
1,"(-8.639847, 41.159826)","(-8.66574, 41.170671)"
2,"(-8.612964, 41.140359)","(-8.61597, 41.14053)"
3,"(-8.574678, 41.151951)","(-8.607996, 41.142915)"
4,"(-8.645994, 41.18049)","(-8.687268, 41.178087)"


# Distancia, duraci?n y velocidad por viaje

La distancia suma las distancias entre puntos consecutivos. Cada punto representa 15 segundos; por tanto, la duraci?n es `(n?mero de puntos - 1) ? 15` segundos. La velocidad media y la m?xima se expresan en km/h. Se invierte el orden de las coordenadas al convertirlas a radianes (`longitud`, `latitud`).

In [19]:
def process_polyline(points):
    coordinates = np.asarray(points, dtype=np.float64)
    if len(coordinates) < 2:
        return np.nan, np.nan, np.nan, np.nan

    lon = np.radians(coordinates[:, 0])
    lat = np.radians(coordinates[:, 1])
    delta_lon = np.diff(lon)
    delta_lat = np.diff(lat)

    haversine = (
        np.sin(delta_lat / 2) ** 2
        + np.cos(lat[:-1]) * np.cos(lat[1:]) * np.sin(delta_lon / 2) ** 2
    )
    segment_distances_km = 2 * 6371.0088 * np.arcsin(np.sqrt(np.clip(haversine, 0, 1)))
    trip_distance_km = float(segment_distances_km.sum())
    trip_time_seconds = (len(coordinates) - 1) * 15
    average_speed_kmh = trip_distance_km / (trip_time_seconds / 3600)
    top_speed_kmh = float(segment_distances_km.max() * 3600 / 15)

    return trip_distance_km, trip_time_seconds / 60, average_speed_kmh, top_speed_kmh

Prueba de las m?tricas sobre el primer viaje v?lido de la muestra:

In [20]:
distance, duration_min, average_speed, top_speed = process_polyline(taxi["POINTS"].iloc[0])
print(
    f"Distancia:       {distance:.2f} km\n"
    f"Duraci?n:        {duration_min:.1f} min\n"
    f"Velocidad media: {average_speed:.1f} km/h\n"
    f"Velocidad m?x.:  {top_speed:.1f} km/h"
)

Distancia:       2.65 km
Duraci?n:        5.5 min
Velocidad media: 28.9 km/h
Velocidad m?x.:  68.2 km/h


In [21]:
def map_trip_metrics(df_in):
    metrics = df_in["POINTS"].map(process_polyline)
    return pd.DataFrame(
        metrics.to_list(),
        columns=["TRIP_DISTANCE", "TRIP_TIME", "AVERAGE_SPEED", "TOP_SPEED"],
        index=df_in.index,
    )

In [22]:
metric_columns = map_trip_metrics(taxi)
taxi = pd.concat([taxi, metric_columns], axis=1)
taxi[["TRIP_DISTANCE", "TRIP_TIME", "AVERAGE_SPEED", "TOP_SPEED"]].describe()

,TRIP_DISTANCE,TRIP_TIME,AVERAGE_SPEED,TOP_SPEED
count,9838.000000,9838.000000,9838.000000,9838.000000
mean,6.286855,12.052323,29.950237,153.100014
std,10.541586,9.893870,23.992706,655.819519
min,0.000000,0.250000,0.000000,0.000000
25%,2.495860,7.000000,18.776955,51.998985
50%,4.131541,10.250000,24.580071,67.996253
75%,7.306075,14.500000,35.325507,112.368553
max,635.263991,230.250000,1070.622245,37519.778084


# Exportaci?n opcional de la muestra

La exportaci?n est? desactivada por defecto. El archivo de salida es solo una muestra de an?lisis, no reemplaza la ingesta completa con Dask.

In [23]:
EXPORT_SAMPLE = False
if EXPORT_SAMPLE:
    output_path = Path("train_sample_extended.csv.gz")
    taxi.drop(columns=["POINTS"]).to_csv(
        output_path,
        index=False,
        compression="gzip",
    )
    print(f"Muestra exportada en: {output_path.resolve()}")